# 02 — Export (Earth Engine → Cloud Storage)

Creates a run, exports **one pilot chunk**, and, after you have checked it, all chunks.

Exports cost Earth Engine quota. **Planning** only writes local rows in state `PLANNED`; nothing is exported until you set a `CONFIRMED…` flag to `True`, which **confirms that scope** (`PLANNED → PENDING`). Only confirmed rows are ever submitted, also by the Monitor cell.

> **Safe to re-run:** finished work is skipped. If the kernel dies or a cell crashes, just run the same cells again.

## Setup

In [ ]:
from pathlib import Path
import sys

# Project root = parent of notebooks/. Adding src/ is only needed if you did not run `pip install -e .`
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

# >>> Change this to YOUR config file (copied from config/pipeline.example.yaml; it must live in config/) <<<
CONFIG_PATH = ROOT / "config" / "my_aoi_season.yaml"

from sar_pipeline import config, resources
user_cfg = config.load_config(ROOT / CONFIG_PATH)   # your editable config; the RUN config is loaded below
print("Config :", CONFIG_PATH)
print("Machine:", resources.detect_resources(user_cfg).describe())

## Step 3 — Run folder

A run is an immutable folder `runs/v<NNN>_<date>/` holding a **frozen copy of your config** and a unique `run_uid.txt` (used in Cloud Storage paths and task names). Fill `s1.tracks` in your config first (notebook 01, Checkpoint 1); otherwise creating a run raises `AuditRequired`.

- First time: set `CREATE_NEW_RUN = True`.
- Continuing (e.g. after a crash, or pilot → full export): keep `False` to reuse the newest run.

**Which config is used?** After a run is chosen, everything uses the run's **frozen** config (`runs/<run_id>/run_config.yaml`), so this run's files are always read with the settings that produced them. If you edited your own config since `new-run`, a warning lists the differing keys; such changes need a **new run**. `auth.project` also stays the run's project. Taken from your config instead: `qa.acknowledged_issues` (accepting QA issues is a decision made after the run), `auth.key_file` and `resources` (they describe the machine, so a run exported on a laptop can be continued on a cloud notebook server).

In [ ]:
CREATE_NEW_RUN = False   # <- True only when you want a NEW run (after filling s1.tracks)

from sar_pipeline.cli import load_run_context, require_tracks

if CREATE_NEW_RUN:
    require_tracks(user_cfg)       # a run needs selected tracks
    config.new_run(user_cfg)       # freezes your config into runs/<run_id>/run_config.yaml

cfg, run_path = load_run_context(user_cfg)   # frozen run config from here on (warns about later edits)
print("Run:", run_path, "| tracks:", [t["track_id"] for t in cfg["s1"]["tracks"]])

## Step 4 — Pilot export plan

Pick a chunk fully inside the AOI (see notebook 01, `aoi_frac` ≈ 1). Planning only writes local files (band layout, `PLANNED` manifest rows); nothing is exported yet. Tracks that never cover the chunk are marked `NOT_COVERED`.

In [ ]:
from sar_pipeline import auth, export, manifest

auth.init_ee(cfg)
PILOT_CHUNK = "chunk_r01c01"   # <- change to your pilot chunk
plan = export.plan_exports(cfg, run_path, chunk_names=[PILOT_CHUNK])
plan[plan["chunk_name"] == PILOT_CHUNK][["task_key", "track_id", "chunk_name", "n_bands", "state"]]

## ⛔ Confirm the pilot export

Check the plan above (one task per selected track that covers the chunk). **Change `CONFIRMED` to `True`** to confirm this pilot scope and start it. With `False`, nothing is confirmed and `submit_pending` raises `ConfirmationRequired`.

In [ ]:
CONFIRMED = False   # <- set to True after checking the plan above

if CONFIRMED:
    export.confirm_exports(cfg, run_path, chunk_names=[PILOT_CHUNK])   # PLANNED -> PENDING for the pilot only
export.submit_pending(cfg, run_path, confirmed=CONFIRMED)
manifest.read_manifest(run_path)["state"].value_counts()

## Monitor

Polls Earth Engine until every **confirmed** task has finished and applies the retry policy (backoff on quota and transient errors, split into 4 sub-chunks on memory/timeout). It may **re-submit** confirmed tasks, so it also needs `CONFIRMED`. It never submits `PLANNED` rows. You can interrupt and re-run this cell any time; only one monitor may run per run.

For long exports, prefer a terminal (logs go to the gitignored `logs/` folder):
`mkdir -p logs && nohup python -m sar_pipeline --config <cfg> monitor --yes > logs/monitor.log 2>&1 &`

In [ ]:
m = export.monitor(cfg, run_path, confirmed=CONFIRMED, until_done=True)
m["state"].value_counts()

In [ ]:
import pandas as pd

failed_report = export.write_failed_report(run_path)
pd.read_csv(failed_report)   # empty = nothing failed; retry deliberately with export.retry_failed(run_path, cfg=cfg)

Now download and inspect the pilot with `03_download_and_stack.ipynb` (**Checkpoint 2** in `docs/04_runbook.md`). Come back here only when the pilot looks right.

## Step 5 — Full export plan

Plans all remaining chunks in the **same run** (the pilot's tasks are kept, not repeated). New rows are `PLANNED` until confirmed below.

In [ ]:
full_plan = export.plan_exports(cfg, run_path)
print(full_plan["state"].value_counts())
print("Planned tasks per track:")
print(full_plan[full_plan["state"] == "PLANNED"].groupby("track_id").size())

## ⛔ Confirm the full export

Check the number of planned tasks above. **Change `CONFIRMED_ALL` to `True`** to confirm every planned chunk and start.

In [ ]:
CONFIRMED_ALL = False   # <- set to True after checking the task counts above

if CONFIRMED_ALL:
    export.confirm_exports(cfg, run_path)          # all PLANNED rows -> PENDING
export.submit_pending(cfg, run_path, confirmed=CONFIRMED_ALL)
m = export.monitor(cfg, run_path, confirmed=CONFIRMED_ALL, until_done=True)
print(m["state"].value_counts())
pd.read_csv(export.write_failed_report(run_path))

## Retry failed exports (only when needed)

After fixing the cause of failures listed in `failed_chunks.csv`, put FAILED rows back into the queue deliberately.

In [ ]:
RETRY_FAILED = False   # <- set to True to re-queue FAILED export rows

if RETRY_FAILED:
    export.retry_failed(run_path, cfg=cfg)
    export.submit_pending(cfg, run_path, confirmed=True)
    m = export.monitor(cfg, run_path, confirmed=True, until_done=True)
    print(m["state"].value_counts())